# RL PoC — TradingMVP

Demuestra que el loop RL funciona end-to-end: entorno gym → política → recompensa → actualización PPO.

**No entrena para aprender** — solo verifica que la plomería es correcta.

In [ ]:
# Celda 1 — Instalación (solo en Colab)
import os
IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ

if IN_COLAB:
    !pip install -q "setuptools==65.5.0"
    !pip install -q coloredlogs==15.0.1 "numpy==1.22.0" "pandas==1.2.4" "scipy==1.10.0" pomegranate==0.14.6 termcolor "gym==0.21.0" torch
    !git clone https://github.com/LuzMRB/TradingMVP.git /content/TradingMVP 2>/dev/null || echo 'ya clonado'
    %cd /content/TradingMVP
    !pip install -q -e abides-jpmc-public/abides-core
    !pip install -q -e abides-jpmc-public/abides-markets
    !pip install -q -e abides-jpmc-public/abides-gym
    import sys
    sys.path.insert(0, '/content/TradingMVP')

print('Entorno listo')

In [ ]:
# Celda 2 — Crear entorno y verificar espacios
from src.env.spy_gym_env import SpyGymEnv

env = SpyGymEnv(
    background_config="rmsc04",
    mkt_close="10:00:00",
    timestep_duration="60s",
    starting_cash=1_000_000,
    order_fixed_size=10,
    first_interval="00:05:00",
)

obs = env.reset()

print(f"observation_space: {env.observation_space}")
print(f"action_space:      {env.action_space}")
print(f"obs shape:         {obs.shape}")   # esperado: (44,)
print(f"obs dtype:         {obs.dtype}")
print(f"obs:               {obs}")
assert obs.shape == (44,), f"Esperado (44,) pero got {obs.shape}"
print("\nOK — shape correcto")

In [ ]:
# Celda 3 — Política aleatoria: 1 episodio completo
import numpy as np

obs = env.reset()
done = False
rewards = []

while not done:
    action = env.action_space.sample()
    obs, reward, done, info = env.step(action)
    rewards.append(reward)

print(f"Steps:         {len(rewards)}")
print(f"Reward total:  {sum(rewards):.4f}")
print(f"Reward medio:  {np.mean(rewards):.6f}")
print("OK — episodio completo sin errores")

In [ ]:
# Celda 4 — Loop PPO (500 steps, no pretende converger)
from src.training.ppo_trainer import PPOTrainer

env2 = SpyGymEnv(
    background_config="rmsc04",
    mkt_close="10:00:00",
    timestep_duration="60s",
    starting_cash=1_000_000,
    order_fixed_size=10,
    first_interval="00:05:00",
)

trainer = PPOTrainer(
    env=env2,
    rollout_length=50,
    batch_size=25,
    update_epochs=2,
    device="cpu",
)

print(f"obs_dim:   {env2.observation_space.shape[0]}")
print(f"n_actions: {env2.action_space.n}")
print(f"red:       {trainer.network}\n")

trainer.train(total_steps=500, log_interval=1)
print("\nOK — loop PPO completado sin errores")

In [ ]:
# Celda 5 — Plot rewards por episodio
import matplotlib.pyplot as plt

rewards_ep = trainer.episode_rewards

if rewards_ep:
    plt.figure(figsize=(10, 4))
    plt.plot(rewards_ep, marker='o', linewidth=1, markersize=4)
    plt.xlabel("Episodio")
    plt.ylabel("Reward total")
    plt.title("PPO PoC — rmsc04 (política aleatoria inicial)")
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    print(f"Episodios: {len(rewards_ep)},  Reward medio: {np.mean(rewards_ep):.2f}")
else:
    print("No se completaron episodios enteros en 500 steps (normal con episodios largos)")
    print(f"Reward acumulado parcial: {trainer.current_episode_reward[0]:.2f}")